In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [2]:
import mlflow

!mlflow db upgrade sqlite:///mlflow.db
mlflow.set_tracking_uri('sqlite:///mlflow.db')

2026/07/11 06:02:01 INFO mlflow.store.db.utils: Updating database tables


In [3]:
ARCHS = sorted(e.name[:-len('_Training')] for e in mlflow.MlflowClient().search_experiments()
              if e.name.endswith('_Training'))

client = mlflow.MlflowClient()

def cv_wmae(arch):
    exp = client.get_experiment_by_name(f'{arch}_Training')
    if exp is None:
        return None
    runs = client.search_runs(exp.experiment_id, filter_string=f"attributes.run_name = '{arch}_CV'",
                              order_by=['attributes.start_time DESC'], max_results=1)
    return runs[0].data.metrics.get('wmae_mean') if runs else None

scores = {arch: score for arch in ARCHS if (score := cv_wmae(arch)) is not None}
best_arch = min(scores, key=scores.get)
scores, best_arch

({'DLinear': 3017.139804240749,
  'DLinear_v2': 3170.6185656180824,
  'LightGBM': 3033.187640134209,
  'LightGBM_v2': 2917.763069440465,
  'Prophet': 19359.97616227788,
  'Prophet_v2': 2610.5097211165476,
  'XGBoost': 3207.5995460363447,
  'XGBoost_v2': 2883.558476219258},
 'Prophet_v2')

In [4]:
MODEL_NAME = 'walmart-sales-model'

exp = client.get_experiment_by_name(f'{best_arch}_Training')
final_run = client.search_runs(exp.experiment_id, filter_string=f"attributes.run_name = '{best_arch}_Final'",
                               order_by=['attributes.start_time DESC'], max_results=1)[0]

registered = [m.name for m in client.search_registered_models()]
current = client.get_registered_model(MODEL_NAME).latest_versions[0] if MODEL_NAME in registered else None
if current is None or current.run_id != final_run.info.run_id:
    mlflow.register_model(f'runs:/{final_run.info.run_id}/model', MODEL_NAME)

client.get_registered_model(MODEL_NAME).latest_versions[0]

Registered model 'walmart-sales-model' already exists. Creating a new version of this model...
2026/07/11 06:02:05 WARNING mlflow.tracking._model_registry.fluent: Run with id b8f7d08287944885b7cd2e6a49bc8602 has no artifacts at artifact path 'model', registering model based on models:/m-ea0dc26570c74a74a5e00350586a9a80 instead
Created version '2' of model 'walmart-sales-model'.


<ModelVersion: aliases=[], creation_timestamp=1783749725958, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1783749725958, metrics=None, model_id=None, name='walmart-sales-model', params=None, run_id='b8f7d08287944885b7cd2e6a49bc8602', run_link=None, source='models:/m-ea0dc26570c74a74a5e00350586a9a80', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [5]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Register best model"
    !git push

[main 4050261] Register best model
 1 file changed, 0 insertions(+), 0 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 585 bytes | 585.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/giomamaca/walmart-sales-forecasting.git
   f72a10a..4050261  main -> main
